In [1]:
%pip install langchain langchain-community langchain-huggingface qdrant-client langchain-qdrant langchain-text-splitters


   -------------------- ------------------- 1/2 [langchain-huggingface]
   -------------------- ------------------- 1/2 [langchain-huggingface]
   ---------------------------------------- 2/2 [langchain-huggingface]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore


file_path = "data/Tesla_report.pdf"

pdf_data = PyPDFLoader(
    file_path=file_path
)

pdf_load = pdf_data.load()

print(f"Document Loaded Succesfully")
print(f"Total Documents : {len(pdf_load)}")

for i,data in enumerate(pdf_load[:3]): # Only get 1st 3 documents
    print(f"Metadata Of the pdf {data.metadata} \n")
    print(f"Page Content of Document \n {data.page_content} " )
    print("="*60)


C:\Users\Muhammad Hamza\AppData\Local\Temp\ipykernel_6980\1633512439.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\Muhammad Hamza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document Loaded Succesfully
Total Documents : 144
Metadata Of the pdf {'producer': 'Qt 5.15.2', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2025-01-30T11:11:07+00:00', 'title': '', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'} 

Page Content of Document 
 UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
FORM	
10-K
(Mark	One)
x
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31
,	2024
OR
o
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	_________	to	_________
Commission	File	Number:	
001-34756
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
Texas
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
(I.R.S.	Employer
Identification	No.)
1	Tesla	Road
Austin
,	
Texas
78725
(Address	of	principal	executive	offices)
(Zip	Code)
(
512
)	
516-8177


In [6]:
#  Make Chunkings of the document 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300
)

chunks = splitter.split_documents(pdf_load)

print(f" Chunking data document total {len(chunks)} ")
print("Medium 3 chunking Data ")

for i, data in enumerate(chunks[45:48]):
    print(i + 1)
    print(f"Chunking meta data {data.metadata} " )
    print(f" Chunking Page content : \n {data.page_content}")
    print("="*60)

 Chunking data document total 561 
Medium 3 chunking Data 
1
Chunking meta data {'producer': 'Qt 5.15.2', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2025-01-30T11:11:07+00:00', 'title': '', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'page': 11, 'page_label': '12'} 
 Chunking Page content : 
 Some	automobile	dealer	trade	associations	have	both	challenged	the	legality	of	our	operations	in	court	and	used	administrative	and	legislative
processes	to	attempt	to	prohibit	or	limit	our	ability	to	operate	existing	stores	or	expand	to	new	locations.	Certain	dealer	associations	have	also	actively
lobbied	state	licensing	agencies	and	legislators	to	interpret	existing	laws	or	enact	new	laws	in	ways	not	favorable	to	our	ownership	and	operation	of	our
own	retail	and	service	locations.	We	expect	such	challenges	to	continue,	and	we	intend	to	actively	fight	any	such	efforts.
Battery	Safety	and	Testing
Our	battery	packs	are	subject	to	various	U.S.	and	international	regulations	that	gove

In [ ]:
# Make embeddings of the chunking data and store this into Qdrant

from qdrant_client import QdrantClient



QDRANT_URL=""
QDRANT_API=""

client = QdrantClient(
    url = QDRANT_URL,
    api_key = QDRANT_API
)

if client.collection_exists(collection_name = "tesla_report"):
    client.delete_collection(collection_name = "tesla_report")
    print("Delete Old Collection Successfully")


embeddings=HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API,
    prefer_grpc=True, 
    collection_name="tesla_report",
    timeout=240
)

print("Document Stored Successfully Stored in Qdrant Cloud Server")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1284.43it/s]


Document Stored Successfully Stored in Qdrant Cloud Server


In [ ]:
#  Check Status of QDRANT cloud
from qdrant_client import QdrantClient

QDRANT_URL=""
QDRANT_API=""

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API
)

get = client.get_collection("tesla_report")

print(f"Cloud Status {get.status} ")
print(f"Total Vectors {get.points_count} ")

Cloud Status green 
Total Vectors 561 


In [10]:
#  Now testing queries

query = "How does Tesla manufacture batteries?"

retriever = vector_store.as_retriever() # Create it in single line
results = retriever.invoke(query)

for i, result in enumerate(results):
    print(f" Index No { i + 1 } \n")
    print(f"Metadata : {result.metadata} \n Page Content of result \n {result.page_content[:50]} ")
    print("="*60)

 Index No 1 

Metadata : {'creationdate': '2025-01-30T11:11:07+00:00', 'producer': 'Qt 5.15.2', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'page': 17, 'page_label': '18', 'title': '', 'creator': 'wkhtmltopdf 0.12.6', '_id': '2f49d055-3584-48b8-9452-3316a91a3e80', '_collection_name': 'tesla_report'} 
 Page Content of result 
 manage	our	growth	effectively,	it	may	harm	our	bra 
 Index No 2 

Metadata : {'creationdate': '2025-01-30T11:11:07+00:00', 'title': '', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'producer': 'Qt 5.15.2', 'page_label': '37', 'page': 36, 'creator': 'wkhtmltopdf 0.12.6', '_id': '0a04629f-a170-439a-ba9c-f7129c019588', '_collection_name': 'tesla_report'} 
 Page Content of result 
 Shanghai	and	Lathrop,	California.	For	Megapack,	en 
 Index No 3 

Metadata : {'creationdate': '2025-01-30T11:11:07+00:00', 'producer': 'Qt 5.15.2', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'page': 17, 'page_label': '18', 'title': '', 'creator': 'wkhtmltopd

In [11]:
#  Lets see MMR Results 

query = "How does tesla Manufecturer Batteries?"

mmr = vector_store.max_marginal_relevance_search(
    query,
    k=4,
    fetch_k=8,
    lambda_mult = 0.5
)

for i, results in enumerate(mmr):
    print(f"Index No : {i +1}")
    print(f"Meta data : {results.metadata} \n Page Content \n {results.page_content[:50]} ")
    print("="*60)

Index No : 1
Meta data : {'creationdate': '2025-01-30T11:11:07+00:00', 'producer': 'Qt 5.15.2', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'page': 17, 'page_label': '18', 'title': '', 'creator': 'wkhtmltopdf 0.12.6', '_id': '4cf91a00-ad53-49ca-97d7-5a1cc77bf954', '_collection_name': 'tesla_report'} 
 Page Content 
 volumes	and	more	cost-effective	than	currently	ava 
Index No : 2
Meta data : {'creationdate': '2025-01-30T11:11:07+00:00', 'producer': 'Qt 5.15.2', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'title': '', 'page_label': '14', 'page': 13, 'creator': 'wkhtmltopdf 0.12.6', '_id': '60f5e6da-5ffe-4416-b510-d2ec49479872', '_collection_name': 'tesla_report'} 
 Page Content 
 Tesla	plans	to	hire	over	1,000	participants	across 
Index No : 3
Meta data : {'page': 140, 'producer': 'Qt 5.15.2', 'source': 'data/Tesla_report.pdf', 'total_pages': 144, 'creationdate': '2025-01-30T11:11:07+00:00', 'page_label': '141', 'title': '', 'creator': 'wkhtmltopdf 0.12.6', '_id': 